In [20]:
import fiftyone as fo
from fiftyone.types import VideoDirectory
from fiftyone import ViewField as F

import easyocr
from PIL import Image
import numpy as np


In [21]:
## Ingest your videos as before
dataset_name = "data_test"
if not fo.dataset_exists(dataset_name):
    dataset = fo.Dataset.from_dir(
        dataset_dir="../../data/test/",
        dataset_type=VideoDirectory,
        name=dataset_name,
        persistent=True
    )
else:
    dataset = fo.load_dataset(dataset_name)

dataset.persistent=True
dataset.ensure_frames()
dataset

Name:        data_test
Media type:  video
Num samples: 11
Persistent:  True
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.VideoMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
Frame fields:
    id:               fiftyone.core.fields.ObjectIdField
    frame_number:     fiftyone.core.fields.FrameNumberField
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    filepath:         fiftyone.core.fields.StringField
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)

In [22]:
dataset.compute_metadata()

In [23]:

## Frame-level view
frames_view = dataset.to_frames(sample_frames=True,  # Enable frame sampling
    fps=1,              # Sample 1 frame per second
    max_fps=10,          # Maximum 1 frame per second
    force_sample=True,  # Force resampling even if frames exist
    verbose=True,        # Show progress
)


Determining frames to sample...
Must sample 284/8508 frames of '/home/mbudisic/Documents/PsTuts-VQA-Dataset/data/test/Add a central element.mp4'
Must sample 298/8926 frames of '/home/mbudisic/Documents/PsTuts-VQA-Dataset/data/test/Add text.mp4'
Must sample 249/7447 frames of '/home/mbudisic/Documents/PsTuts-VQA-Dataset/data/test/Adjust brightness and contrast.mp4'
Must sample 246/3685 frames of '/home/mbudisic/Documents/PsTuts-VQA-Dataset/data/test/Get organized with layer groups.mp4'
Must sample 263/7883 frames of '/home/mbudisic/Documents/PsTuts-VQA-Dataset/data/test/Include vector graphics.mp4'
Must sample 192/5760 frames of '/home/mbudisic/Documents/PsTuts-VQA-Dataset/data/test/Remove a large object.mp4'
Must sample 315/9440 frames of '/home/mbudisic/Documents/PsTuts-VQA-Dataset/data/test/Remove unwanted content.mp4'
Must sample 84/2502 frames of '/home/mbudisic/Documents/PsTuts-VQA-Dataset/data/test/Remove unwanted objects from photos.mp4'
Must sample 307/9181 frames of '/home/mbu

In [24]:
frames_view.compute_metadata()

Computing metadata...
 100% |███████████████| 2760/2760 [373.3ms elapsed, 0s remaining, 7.4K samples/s]      


In [25]:
fo.launch_app(frames_view)

Dataset:          data_test
Media type:       video
Num samples:      2760
Selected samples: 0
Selected labels:  0
Session URL:      http://localhost:5151/
View stages:
    1. ToFrames(config={'force_sample': True, 'fps': 1, 'max_fps': 10, ...})

In [26]:
import os
from getpass import getpass
from dotenv import load_dotenv
load_dotenv()

def set_api_key_if_not_present(key_name, prompt_message=""):
    if len(prompt_message) == 0:
        prompt_message=key_name
    if key_name not in os.environ or not os.environ[key_name]:
        os.environ[key_name] = getpass.getpass(prompt_message)
        
set_api_key_if_not_present("OPENAI_API_KEY")

In [27]:

## Build keyframe view (one frame per second)
keyframes = frames_view.match(
    F("frame_number") % F("metadata.fps") == 0
)
len(keyframes)

0

In [28]:
import easyocr 

In [29]:
len(keyframes)

0

In [30]:
import torch
import torchvision


In [31]:

## Initialize EasyOCR reader (download models on first run)
reader = easyocr.Reader(["en"],gpu=torch.cuda.is_available())  # specify any languages you need


In [32]:

## OCR each keyframe
for frame in keyframes:
    # Convert frame to numpy array if needed
    img_np = frame.frame  # already an H×W×3 uint8 array
    # EasyOCR wants either filepath or numpy array in RGB
    # If you're using PIL, convert back:
    # img_np = np.array(Image.fromarray(frame.frame))
    
    # Run OCR; detail=0 returns just the text strings
    results = reader.readtext(img_np, detail=0, paragraph=True)
    text = "\n".join(results)

    print(f"✂── Video: {frame.metadata.name}   Frame: {frame.frame_number}")
    print(text)
    print("-" * 80)


In [33]:
fo.launch_app()

Dataset:     -
Session URL: http://localhost:5151/